# Volvo hackathon: what you can do with your `.env`

This notebook walks through every capability your team's `.env` unlocks - **no Azure sign-in**,
just the keys and the deploy identity in that file. Run the cells top to bottom.

Each section is one operation:
1. Call the chat model (inference)
2. Create embeddings
3. Search the fact sheets (three vector indexes: all / diesel / electric)
4. Create your **own** search index
5. Cosmos DB read/write
6. Blob storage read
7. Build a container image locally
8. Push it to your Azure Container Registry
9. Deploy it to Azure Container Apps and get the **public URL**
10. Test the deployed app
11. Clean up
12. Bonus (reference examples only, nothing runs): reindex after adding or removing files

**Prerequisites:** your team `.env` next to this notebook, the repo `.venv`
(`pip install -r requirements.txt`), and Docker Desktop running (for sections 7-9).

In [ ]:
# --- Setup: trust the OS cert store (corporate proxy) and load the team .env ---
import os, httpx
from pathlib import Path
from dotenv import load_dotenv

# Same TLS handling as the app: trust the machine cert store so a corporate proxy does not break HTTPS.
try:
    import truststore
    truststore.inject_into_ssl()
except Exception:
    pass

ENV_PATH = Path.cwd() / ".env"
load_dotenv(ENV_PATH)

def cfg(name, default=""):
    return (os.getenv(name, default) or "").strip()

# Last-resort fallback if truststore cannot satisfy the proxy (same switch as the app/verify).
INSECURE_SSL = cfg("INSECURE_SSL").lower() in ("1", "true", "yes")
VERIFY = not INSECURE_SSL

AOAI_ENDPOINT = cfg("AZURE_OPENAI_ENDPOINT").rstrip("/")
AOAI_KEY = cfg("AZURE_OPENAI_KEY")
AOAI_API_VERSION = cfg("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_DEPLOYMENT = cfg("CHAT_DEPLOYMENT", "gpt-5.6-luna")
EMBEDDING_DEPLOYMENT = cfg("EMBEDDING_DEPLOYMENT", "text-embedding-3-large")

SEARCH_ENDPOINT = cfg("SEARCH_ENDPOINT").rstrip("/")
SEARCH_KEY = cfg("SEARCH_KEY")
SEARCH_INDEX = cfg("SEARCH_INDEX", "allfactsheets-index")

COSMOS_ENDPOINT = cfg("COSMOS_ENDPOINT")
COSMOS_KEY = cfg("COSMOS_KEY")
COSMOS_DATABASE = cfg("COSMOS_DATABASE", "truckoffer")

STORAGE_ACCOUNT = cfg("STORAGE_ACCOUNT")
STORAGE_CONTAINER = cfg("STORAGE_CONTAINER", "hackdata")
STORAGE_SAS = cfg("STORAGE_SAS").lstrip("?")

ACR_LOGIN_SERVER = cfg("ACR_LOGIN_SERVER")
ACR_USERNAME = cfg("ACR_USERNAME")
ACR_PASSWORD = cfg("ACR_PASSWORD")

AZURE_TENANT_ID = cfg("AZURE_TENANT_ID")
AZURE_CLIENT_ID = cfg("AZURE_CLIENT_ID")
AZURE_CLIENT_SECRET = cfg("AZURE_CLIENT_SECRET")
AZURE_SUBSCRIPTION_ID = cfg("AZURE_SUBSCRIPTION_ID")
AZURE_RESOURCE_GROUP = cfg("AZURE_RESOURCE_GROUP")
CONTAINER_APPS_ENVIRONMENT = cfg("CONTAINER_APPS_ENVIRONMENT")

assert AOAI_ENDPOINT and AOAI_KEY, "AZURE_OPENAI_* missing - is the .env next to this notebook?"
print("Loaded:", ENV_PATH)
print("Foundry:", AOAI_ENDPOINT)
print("Search :", SEARCH_ENDPOINT, "| index:", SEARCH_INDEX)
print("Cosmos :", COSMOS_ENDPOINT, "| db:", COSMOS_DATABASE)
print("Deploy :", AZURE_RESOURCE_GROUP, "| ACA env:", CONTAINER_APPS_ENVIRONMENT)
print("TLS verify:", VERIFY)

## 1. Call the chat model (inference)
Uses the Azure OpenAI-compatible endpoint with your key.

In [ ]:
from openai import AzureOpenAI

aoai = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
    api_version=AOAI_API_VERSION,
    http_client=httpx.Client(verify=VERIFY),
)

resp = aoai.chat.completions.create(
    model=CHAT_DEPLOYMENT,
    messages=[{"role": "user", "content": "In one sentence, what is a Volvo FH truck used for?"}],
    max_completion_tokens=400,
)
print(resp.choices[0].message.content)

## 2. Create embeddings
Turn text into a vector with your embedding deployment.

In [ ]:
emb = aoai.embeddings.create(model=EMBEDDING_DEPLOYMENT, input="Volvo FH truck fuel efficiency features")
vector = emb.data[0].embedding
print("dimensions:", len(vector))
print("first 5 values:", [round(v, 4) for v in vector[:5]])

## 3. Search the fact sheets
Three vector indexes were built from the Volvo fact sheets, all queryable with your `SEARCH_KEY`:
- `allfactsheets-index` - every fact sheet (the default `SEARCH_INDEX`)
- `diesel-truck-index` - diesel specs
- `electric-truck-index` - electric specs

Hybrid (keyword + vector) query; each index embeds your query server-side, so you just pass text.

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

query = "fuel efficiency"
indexes = {
    "all":      SEARCH_INDEX,
    "diesel":   cfg("SEARCH_INDEX_DIESEL", "diesel-truck-index"),
    "electric": cfg("SEARCH_INDEX_ELECTRIC", "electric-truck-index"),
}
for label, ix in indexes.items():
    search = SearchClient(SEARCH_ENDPOINT, ix, AzureKeyCredential(SEARCH_KEY), connection_verify=VERIFY)
    results = search.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=3, fields="content_vector")],
        select=["title", "content"],
        top=2,
    )
    print(f"=== {label}: {ix} ===")
    for r in results:
        print("-", r["title"], "| score", round(r["@search.score"], 3))

## 4. Create your own search index
Define a small index, upload documents, and query it.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchableField, SearchFieldDataType

index_admin = SearchIndexClient(SEARCH_ENDPOINT, AzureKeyCredential(SEARCH_KEY), connection_verify=VERIFY)
DEMO_INDEX = "notebook-demo"

index_admin.create_or_update_index(SearchIndex(name=DEMO_INDEX, fields=[
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
]))
print("created index:", DEMO_INDEX)

demo = SearchClient(SEARCH_ENDPOINT, DEMO_INDEX, AzureKeyCredential(SEARCH_KEY), connection_verify=VERIFY)
demo.upload_documents([
    {"id": "1", "title": "Volvo FH",  "content": "Long-haul flagship with I-Save fuel economy."},
    {"id": "2", "title": "Volvo FMX", "content": "Construction truck with a heavy-duty chassis."},
])

import time; time.sleep(2)  # let indexing settle
for r in demo.search(search_text="construction"):
    print("hit:", r["title"], "-", r["content"])

## 5. Cosmos DB
Create a container, write, read, and query - all with the account key.

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey

cosmos = CosmosClient(COSMOS_ENDPOINT, COSMOS_KEY, connection_verify=VERIFY)
db = cosmos.create_database_if_not_exists(COSMOS_DATABASE)
container = db.create_container_if_not_exists(id="notebook-demo", partition_key=PartitionKey(path="/id"))

container.upsert_item({"id": "demo-1", "kind": "note", "text": "hello from the notebook"})
print("read back:", container.read_item("demo-1", partition_key="demo-1")["text"])
count = list(container.query_items("SELECT VALUE COUNT(1) FROM c", enable_cross_partition_query=True))
print("items in container:", count[0])

## 6. Blob storage
List and download from the hackathon data container using the SAS token (no extra SDK).

In [ ]:
import re, html, urllib.parse

blob_base = f"https://{STORAGE_ACCOUNT}.blob.core.windows.net"
headers = {"x-ms-version": "2021-08-06"}

list_url = f"{blob_base}/{STORAGE_CONTAINER}?restype=container&comp=list&{STORAGE_SAS}"
xml = httpx.get(list_url, headers=headers, verify=VERIFY, timeout=60).text
names = [html.unescape(n) for n in re.findall(r"<Name>(.*?)</Name>", xml)][:5]
print("first blobs:", names)

if names:
    read_url = f"{blob_base}/{STORAGE_CONTAINER}/{urllib.parse.quote(names[0])}?{STORAGE_SAS}"
    data = httpx.get(read_url, headers=headers, verify=VERIFY, timeout=60).content
    print("downloaded", names[0], "-", len(data), "bytes")

## 7. Build a container image locally
Sections 7-9 need **Docker Desktop running**. We generate a tiny web app, build it, push it to your
registry, and deploy it to Azure Container Apps.

In [ ]:
import subprocess, tempfile, shutil
from pathlib import Path

DOCKER = shutil.which("docker") or "docker"
build_dir = Path(tempfile.mkdtemp(prefix="hackapp_"))

(build_dir / "app.py").write_text(
    'from http.server import BaseHTTPRequestHandler, HTTPServer\n'
    'class H(BaseHTTPRequestHandler):\n'
    '    def do_GET(self):\n'
    '        self.send_response(200); self.send_header("Content-Type", "text/plain"); self.end_headers()\n'
    '        self.wfile.write(b"Hello from the Volvo hack container!")\n'
    'print("listening on :8000")\n'
    'HTTPServer(("0.0.0.0", 8000), H).serve_forever()\n'
)
(build_dir / "Dockerfile").write_text(
    'FROM python:3.12-slim\nWORKDIR /app\nCOPY app.py .\nEXPOSE 8000\nCMD ["python", "app.py"]\n'
)

IMAGE = f"{ACR_LOGIN_SERVER}/hack-demo:latest"
print("building", IMAGE)
# Azure Container Apps runs linux/amd64; pin it so the image runs regardless of your machine's arch.
subprocess.run([DOCKER, "build", "--platform", "linux/amd64", "--provenance=false", "-t", IMAGE, str(build_dir)], check=True)

## 8. Push to Azure Container Registry
Log in with the ACR admin credentials from your `.env`, then push.

In [ ]:
subprocess.run([DOCKER, "login", ACR_LOGIN_SERVER, "-u", ACR_USERNAME, "--password-stdin"],
               input=ACR_PASSWORD, text=True, check=True)
subprocess.run([DOCKER, "push", IMAGE], check=True)
print("pushed", IMAGE)

## 9. Deploy to Azure Container Apps
Sign in with your team's **service principal** (from the `.env`) into an isolated CLI context, then
create the app in your pre-provisioned environment and read back the public URL.

In [ ]:
import os

# Isolated az context so this does not touch any other CLI login on the machine.
AZ_ENV = {**os.environ, "AZURE_CONFIG_DIR": str(build_dir / ".azure")}

def az(*args, capture=True):
    cmd = "az " + " ".join(args) if os.name == "nt" else ["az", *args]
    r = subprocess.run(cmd, shell=(os.name == "nt"), env=AZ_ENV, check=True, text=True,
                       capture_output=capture)
    return (r.stdout or "").strip()

az("login", "--service-principal", "-u", AZURE_CLIENT_ID, "-p", AZURE_CLIENT_SECRET,
   "--tenant", AZURE_TENANT_ID, "--only-show-errors")
az("account", "set", "--subscription", AZURE_SUBSCRIPTION_ID)
az("extension", "add", "-n", "containerapp", "-y", "--only-show-errors")

APP_NAME = "hack-demo"
az("containerapp", "create", "-n", APP_NAME, "-g", AZURE_RESOURCE_GROUP,
   "--environment", CONTAINER_APPS_ENVIRONMENT, "--image", IMAGE,
   "--registry-server", ACR_LOGIN_SERVER, "--registry-username", ACR_USERNAME,
   "--registry-password", ACR_PASSWORD, "--target-port", "8000", "--ingress", "external",
   "--only-show-errors")

fqdn = az("containerapp", "show", "-n", APP_NAME, "-g", AZURE_RESOURCE_GROUP,
          "--query", "properties.configuration.ingress.fqdn", "-o", "tsv")
URL = "https://" + fqdn
print("Published URL:", URL)

## 10. Test the deployed app
Give ingress a few seconds, then call the public URL (retries while the container cold-starts).

In [ ]:
import time
response = None
for attempt in range(12):
    try:
        r = httpx.get(URL, verify=VERIFY, timeout=30)
        if r.status_code < 500:
            response = r
            break
    except Exception as e:
        response = e
    time.sleep(10)

if hasattr(response, "status_code"):
    print(response.status_code, "->", response.text)
else:
    print("no healthy response yet:", response)

## 11. Clean up
Remove the demo artifacts so nothing lingers (safe to skip while iterating).

In [ ]:
# Container app (stop paying for it)
az("containerapp", "delete", "-n", APP_NAME, "-g", AZURE_RESOURCE_GROUP, "--yes", "--only-show-errors")
# Demo search index + Cosmos container
try: index_admin.delete_index(DEMO_INDEX)
except Exception as e: print("index cleanup:", e)
try: db.delete_container("notebook-demo")
except Exception as e: print("cosmos cleanup:", e)
print("cleaned up demo artifacts")

## 12. Bonus: reindex after adding or removing files
The three fact-sheet indexes are fed by blob **indexers**. These are **reference examples only -
nothing here runs** (they act on the shared indexes, so copy them out when you actually need them).

If you **add or change** files in your storage folders (`factsheets/all`, `factsheets/diesel`,
`factsheets/electric`), just re-run the matching indexer - it tracks blob `LastModified` and only
processes new/changed files:

```python
import time, httpx
api = "2024-07-01"
indexer = "allfactsheets-indexer"   # or diesel-truck-indexer / electric-truck-indexer
sc = httpx.Client(verify=VERIFY, headers={"api-key": SEARCH_KEY}, timeout=60)

sc.post(f"{SEARCH_ENDPOINT}/indexers/{indexer}/run?api-version={api}")
for _ in range(30):
    time.sleep(8)
    last = (sc.get(f"{SEARCH_ENDPOINT}/indexers/{indexer}/status?api-version={api}").json().get("lastResult") or {})
    if last.get("status") and last["status"] != "inProgress":
        print("indexer:", last["status"], "| new/changed processed:", last.get("itemsProcessed"))
        break
count = sc.get(f"{SEARCH_ENDPOINT}/indexes/{SEARCH_INDEX}/docs/$count?api-version={api}").text
print(SEARCH_INDEX, "now has", count, "chunks")
```

**Removing files?** A plain re-run leaves a deleted file's chunks behind (the datasource has no
deletion policy). To reflect removals, **rebuild** the index from the current blobs. The repo ships
`reindex.ps1` that does both:

```powershell
pwsh ./reindex.ps1                    # re-run all three indexers (adds/changes)
pwsh ./reindex.ps1 -Rebuild           # drop + recreate + run from current blobs (adds AND removes)
pwsh ./reindex.ps1 -Rebuild -Index diesel-truck-index
```